# 05 – Bayes-hálók: predikció, diagnózis és FTA → BN gondolkodás

**5. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- Bayes-tétel numerikus alkalmazása
- prior–likelihood–posterior elkülönítése
- prediktív és diagnosztikai következtetés
- TK-101 szenzorevidenciával történő risk update

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Miért Bayes?

Az FTA tipikusan előrefelé kérdez: basic eventekből mekkora a top event valószínűsége? Üzem közben gyakran fordított a kérdés: **evidenciát láttam; mekkora most egy fault valószínűsége?** Ez diagnosztikai inference.


## 2. Bayes-tétel kézzel

$$P(F|E)=\frac{P(E|F)P(F)}{P(E|F)P(F)+P(E|\neg F)P(\neg F)}.$$

Legyen $P(F)=0.02$, detector sensitivity 0.90 és false-positive rate 0.05.


In [ ]:
prior=.02; sens=.90; fp=.05
posterior=sens*prior/(sens*prior+fp*(1-prior))
print(f"Prior={prior:.3f}; posterior={posterior:.3f}")


A posterior közel 27%. A pozitív jel jelentősen növeli a fault valószínűségét, de nem teszi bizonyossá. A base rate számít.


In [ ]:
priors=np.linspace(.001,.2,100)
posts=sens*priors/(sens*priors+fp*(1-priors))
plt.plot(priors,posts); plt.plot(priors,priors,'--'); plt.xlabel('prior'); plt.ylabel('posterior'); plt.grid(alpha=.3); plt.show()


## 3. Prediktív vs diagnosztikai irány

Prediktív: $P(E|F)$ – ha fault van, mit várunk látni? Diagnosztikai: $P(F|E)$ – ha evidenciát látunk, mekkora fault valószínűség? A kettő általában nem azonos.


## 4. Mini Bayes-háló enumerációval

`SensorFault → BadMeasurement → MissedIntervention`. A joint eloszlás faktorizálható: $P(F,M,I)=P(F)P(M|F)P(I|M)$.


In [ ]:
P_F=.02
P_M_given_F={False:.03,True:.90}
P_I_given_M={False:.04,True:.70}
def joint(f,m,i):
    pf=P_F if f else 1-P_F
    pm=P_M_given_F[f] if m else 1-P_M_given_F[f]
    pi=P_I_given_M[m] if i else 1-P_I_given_M[m]
    return pf*pm*pi
rows=[]
for f in [False,True]:
 for m in [False,True]:
  for i in [False,True]: rows.append({'F':f,'M':m,'I':i,'P':joint(f,m,i)})
jt=pd.DataFrame(rows); print('sum=',jt.P.sum()); jt


In [ ]:
subset=jt[jt.M]
print('P(F=True | M=True)=',subset[subset.F].P.sum()/subset.P.sum())


## 5. FTA → BN gondolkodás

FTA gate-ek determinisztikus vagy közel determinisztikus CPT-kkel reprezentálhatók. A BN hozzáadott értéke: evidenciát rögzíthetünk köztes változón, majd diagnosztikai inference-t végzünk. Nem minden FTA→BN konverzió triviális shared events és függőségek mellett.


## 6. TK-101: posterior szenzorhiba → dinamikusan módosított overflow risk

Az FTA `MANUAL_FAIL = LT101_LOW OR OPERATOR_MISS`. Ha a statikus `P(LT101_LOW)=0.02` helyére a diagnosztikai posterior kerül, a top-event probability evidenciától függővé válik.


In [ ]:
p_ch=.08; p_auto=1-(1-.01)*(1-.005); p_op=.04
def overflow(p_sensor):
    p_manual=1-(1-p_sensor)*(1-p_op)
    return p_ch*p_auto*p_manual
print('prior risk    ',overflow(prior))
print('evidence risk ',overflow(posterior))


## 7. Kalibráció

A sensitivity, false-positive rate és prior nem tetszőleges. Forrásuk lehet proof-test, historikus adat, szakértői elicitation vagy validált diagnosztikai kísérlet. A posterior csak annyira megbízható, mint ezek az inputok.


In [ ]:
from safetycourse.bayes import BinaryNode,BinaryBayesNet
nodes=[BinaryNode('Fault',[],{():.02}),BinaryNode('Evidence',['Fault'],{(False,):.05,(True,):.90})]
bn=BinaryBayesNet(nodes)
print(bn.query('Fault',{'Evidence':True}))


## Próbáld ki!

1. Csökkentsd a sensitivity-t 0.70-re.
2. Növeld a false-positive rate-et 0.15-re.
3. Számítsd P(F|M=False)-t enumerációval.
4. Adj MaintenanceOverdue változót, amely a prior fault probability-t befolyásolja, és jelöld egyértelműen modellfeltevésként.


## Összefoglalás

A Bayes-háló a statikus prior modellt evidenciával frissíthető diagnosztikai modellé teszi. A következő tanegységben az **idő és állapotátmenet** kerül a középpontba Markov-modellekkel és Dynamic HAZOP-pal.
